# Logistic Regression on TF-IDF Features

This notebook trains and evaluates a Logistic Regression classifier using the precomputed TF-IDF matrix.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

In [2]:
from pathlib import Path
cwd = Path.cwd().resolve()

# Search upward so this notebook works from project root, notebooks/, or notebooks/experiments/.
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "data").exists() and (candidate / "requirements.txt").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not resolve project root from {cwd}. "
        "Open the notebook from within the hate-speech-detection workspace."
    )

TFIDF_DIR = PROJECT_ROOT / "data" / "processed" / "tfidf"
X_PATH = TFIDF_DIR / "X_tfidf.npz"
Y_PATH = TFIDF_DIR / "y_labels.csv"
if not X_PATH.exists() or not Y_PATH.exists():
    raise FileNotFoundError(
        "Missing TF-IDF artifacts. Run these first:\n"
        "1) .\\.venv\\Scripts\\python.exe scripts/preprocess_data.py\n"
        "2) .\\.venv\\Scripts\\python.exe scripts/vectorize_tfidf.py"
    )

In [3]:
X = load_npz(X_PATH)
y_df = pd.read_csv(Y_PATH)
y = y_df["class"].to_numpy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("Class distribution:")
print(pd.Series(y).value_counts().sort_index())

X shape: (24783, 20000)
y shape: (24783,)
Class distribution:
0     1430
1    19190
2     4163
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (19826, 20000), Test: (4957, 20000)


In [5]:
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
    n_jobs=None,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [6]:
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
}

print("Evaluation metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Evaluation metrics:
accuracy: 0.8679
precision_macro: 0.6955
recall_macro: 0.7937
f1_macro: 0.7330
f1_weighted: 0.8773

Classification report:
              precision    recall  f1-score   support

           0     0.3592    0.5664    0.4396       286
           1     0.9649    0.8747    0.9176      3838
           2     0.7624    0.9400    0.8419       833

    accuracy                         0.8679      4957
   macro avg     0.6955    0.7937    0.7330      4957
weighted avg     0.8960    0.8679    0.8773      4957



In [7]:
labels = np.sort(np.unique(y))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df

,pred_0,pred_1,pred_2
true_0,162,91,33
true_1,270,3357,211
true_2,19,31,783


## Notes

- This baseline uses class-balanced logistic regression.
- You can tune `C`, `ngram_range`, and `max_features` for better results.
- If class `0` (hate speech) recall is low, try stronger class weighting or threshold tuning.